# AQI Data Preprocessing

This notebook prepares the historical AQI dataset for exploratory data analysis and machine learning.

The preprocessing workflow includes datetime conversion, chronological sorting, duplicate handling, missing-value treatment, and statistical outlier inspection.

In [1]:
import numpy as np
import pandas as pd

## 1. Load Raw Historical Data

The historical air-quality observations collected from the OpenWeather API are loaded from the raw data directory.

In [2]:
df = pd.read_csv("../data/raw/aqi_raw_data.csv")

print("Raw dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Raw dataset shape: (120, 10)

Columns:
['datetime', 'AQI', 'CO', 'NO', 'NO2', 'O3', 'SO2', 'PM2_5', 'PM10', 'NH3']


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00


## 2. Datetime Conversion and Chronological Ordering

The datetime column is converted into pandas datetime format and the observations are sorted chronologically.

Maintaining chronological order is important because the later forecasting stage uses temporal and lag-based features.

In [3]:
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

df = (
    df
    .dropna(subset=["datetime"])
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("Dataset shape after datetime validation:", df.shape)

print("\nDate range:")
print(df["datetime"].min())
print(df["datetime"].max())

Dataset shape after datetime validation: (120, 10)

Date range:
2026-08-30 03:00:00
2026-09-04 02:00:00


## 3. Duplicate Observation Check

Duplicate timestamps can create repeated observations and may distort statistical analysis and model training.

Duplicate timestamps are therefore identified and removed.

In [4]:
duplicate_count = df["datetime"].duplicated().sum()

print("Duplicate timestamps:", duplicate_count)

df = (
    df
    .drop_duplicates(subset=["datetime"])
    .sort_values("datetime")
    .reset_index(drop=True)
)

print("Shape after duplicate removal:", df.shape)

Duplicate timestamps: 0
Shape after duplicate removal: (120, 10)


## 4. Missing-Value Analysis

Missing values are examined across all variables.

For numerical pollutant measurements, time-aware interpolation is used where appropriate so that the chronological structure of the dataset is preserved.

In [5]:
print("Missing values before treatment:")
print(df.isnull().sum())

numeric_columns = [
    "AQI",
    "CO",
    "NO",
    "NO2",
    "O3",
    "SO2",
    "PM2_5",
    "PM10",
    "NH3"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[numeric_columns] = (
    df[numeric_columns]
    .interpolate(method="linear", limit_direction="both")
)

print("\nMissing values after treatment:")
print(df.isnull().sum())

Missing values before treatment:
datetime    0
AQI         0
CO          0
NO          0
NO2         0
O3          0
SO2         0
PM2_5       0
PM10        0
NH3         0
dtype: int64

Missing values after treatment:
datetime    0
AQI         0
CO          0
NO          0
NO2         0
O3          0
SO2         0
PM2_5       0
PM10        0
NH3         0
dtype: int64


## 5. Statistical Outlier Inspection

Outliers are identified using the Interquartile Range (IQR) method.

Detected observations are not automatically deleted because extreme pollution measurements may represent genuine air-quality events. The purpose of this step is to identify and inspect unusual observations before modelling.

In [6]:
outlier_summary = []

for column in numeric_columns:

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    )

    outlier_summary.append({
        "Feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outliers.sum())
    })

outlier_df = pd.DataFrame(outlier_summary)

display(outlier_df)

,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,AQI,1.0000,1.0000,0.0000,1.00000,1.00000,6
1,CO,70.2350,74.2650,4.0300,64.19000,80.31000,0
2,NO,0.0000,0.1400,0.1400,-0.21000,0.35000,0
3,NO2,0.9175,1.4500,0.5325,0.11875,2.24875,0
4,O3,34.1175,41.6225,7.5050,22.86000,52.88000,0
5,SO2,0.8600,1.0200,0.1600,0.62000,1.26000,2
6,PM2_5,4.5300,6.0025,1.4725,2.32125,8.21125,0
7,PM10,11.8125,15.3325,3.5200,6.53250,20.61250,5
8,NH3,0.0000,0.0100,0.0100,-0.01500,0.02500,0


## 6. Final Data Quality Verification

The processed dataset is checked for missing values, duplicate timestamps, data types, and chronological ordering before it is saved for feature engineering.

In [7]:
print("Final processed dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate timestamps:")
print(df["datetime"].duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nDate range:")
print(df["datetime"].min())
print(df["datetime"].max())

display(df.head())
display(df.tail())

Final processed dataset shape: (120, 10)

Missing values:
datetime    0
AQI         0
CO          0
NO          0
NO2         0
O3          0
SO2         0
PM2_5       0
PM10        0
NH3         0
dtype: int64

Duplicate timestamps:
0

Data types:
datetime    datetime64[us]
AQI                  int64
CO                 float64
NO                 float64
NO2                float64
O3                 float64
SO2                float64
PM2_5              float64
PM10               float64
NH3                float64
dtype: object

Date range:
2026-08-30 03:00:00
2026-09-04 02:00:00


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
0,2026-08-30 03:00:00,2,70.64,0.05,1.18,35.02,0.90,6.99,20.53,0.00
1,2026-08-30 04:00:00,1,71.37,0.13,1.16,36.15,0.96,6.50,18.09,0.00
2,2026-08-30 05:00:00,1,71.14,0.17,0.98,38.45,0.94,6.11,15.69,0.00
3,2026-08-30 06:00:00,1,70.22,0.16,0.77,41.43,0.91,6.07,14.75,0.01
4,2026-08-30 07:00:00,1,70.41,0.11,0.71,43.64,0.87,6.01,15.01,0.00


,datetime,AQI,CO,NO,NO2,O3,SO2,PM2_5,PM10,NH3
115,2026-09-03 22:00:00,1,69.87,0.00,0.96,34.62,0.78,5.28,13.57,0.0
116,2026-09-03 23:00:00,1,70.22,0.00,0.89,35.06,0.74,5.23,13.99,0.0
117,2026-09-04 00:00:00,1,70.86,0.00,0.89,35.42,0.74,5.20,14.37,0.0
118,2026-09-04 01:00:00,1,71.96,0.00,1.02,35.57,0.78,5.18,14.70,0.0
119,2026-09-04 02:00:00,1,74.20,0.01,1.39,35.42,0.91,5.16,14.89,0.0


## 7. Save Processed Dataset

The cleaned historical AQI dataset is saved in the processed data directory and will be used by the feature-engineering stage.

In [8]:
import os

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/aqi_processed_data.csv"

df.to_csv(output_path, index=False)

print("Processed dataset saved successfully.")
print("Path:", output_path)
print("Final shape:", df.shape)

Processed dataset saved successfully.
Path: ../data/processed/aqi_processed_data.csv
Final shape: (120, 10)


## Conclusion

The historical AQI dataset has been converted into a structured chronological format.

The processed dataset is now ready for time-based feature engineering and subsequent forecasting analysis.